# Phase 3: Chromosome Architecture - Exploration Notebook

This notebook demonstrates:
1. Chromosome initialization and structure
2. Signal generation from a chromosome
3. Genetic operators (crossover, mutation)
4. Fitness evaluation
5. Example evolution loop

**Prerequisites:**
- Phase 2 features pre-computed
- Historical M1 data with all 45 features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import copy
import random
import warnings
warnings.filterwarnings('ignore')

# Import our custom classes
from phase3_Chromosome_Architecture import (
    Gene,
    MultiGene,
    Chromosome,
    TradeSignal,
    initialize_population,
    crossover_uniform,
    mutate,
    fitness,
    select_parents,
    repair,
    validate_chromosome,
    validate_trade_signal
)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports successful")

---
## 1. Load Pre-Computed Features

Assumes you have a parquet/csv file with:
- OHLCV data
- All 45 features from Phase 2 (normalized to [-1, 1])
- Multiple assets columns

In [ ]:
# Load data (adjust path)
# Expected columns: timestamp, open, high, low, close, volume, 
#                   ret_1, ret_5, ..., all Phase 2 features

# For demonstration, create synthetic data
def generate_synthetic_features(n_bars=5000, n_assets=5):
    """
    Generate synthetic normalized features for testing
    """
    np.random.seed(42)
    
    timestamps = pd.date_range(start='2024-01-01', periods=n_bars, freq='1min')
    
    # Base price data
    data = {
        'timestamp': timestamps,
        'open': 1.08 + np.cumsum(np.random.randn(n_bars) * 0.0001),
        'high': np.nan,
        'low': np.nan,
        'close': np.nan,
        'volume': np.random.lognormal(10, 0.5, n_bars),
        'atr': np.random.uniform(0.0001, 0.0003, n_bars)
    }
    
    df = pd.DataFrame(data)
    df['high'] = df['open'] + np.random.uniform(0, 0.0002, n_bars)
    df['low'] = df['open'] - np.random.uniform(0, 0.0002, n_bars)
    df['close'] = df['open'] + np.random.randn(n_bars) * 0.0001
    
    # Generate 45 normalized features (Phase 2 catalogue)
    feature_names = [
        'ret_1', 'ret_5', 'ret_15', 'ret_60',
        'bar_range', 'body_pct', 'wick_upper', 'wick_lower', 'vwap_dev', 'price_pos',
        'rsi_5', 'rsi_14', 'rsi_20', 'roc_5', 'roc_15', 'mom_accel', 'mom_regime', 'trend_strength',
        'vol_regime', 'bb_squeeze', 'hvol_rank', 'vol_accel', 'vol_ratio',
        'hh_score', 'll_score', 'inside_bar', 'engulf_score', 'pivot_prox', 'swing_pos',
        'vol_shock', 'vol_trend', 'vol_sync', 'cvd_delta', 'spread_rank',
        'corr_5', 'corr_15', 'corr_60', 'corr_accel', 'lead_lag_max', 'sync_score',
        'rel_ret_5', 'ratio_z', 'ratio_mom', 'rel_strength',
        'dispersion', 'leadership'
    ]
    
    # All features normalized to [-1, 1] with realistic autocorrelation
    for feat in feature_names:
        base = np.random.randn(n_bars) * 0.3
        # Add autocorrelation
        ar_coef = 0.7
        for i in range(1, n_bars):
            base[i] = ar_coef * base[i-1] + (1 - ar_coef) * base[i]
        df[feat] = np.tanh(base)  # bounded to [-1, 1]
    
    # Add lagged versions (t-1, t-2, t-5, t-15, t-60)
    for feat in feature_names:
        for lag in [1, 2, 5, 15, 60]:
            df[f'{feat}_lag{lag}'] = df[feat].shift(lag)
    
    # Fill NaNs in first rows
    df = df.fillna(0)
    
    return df

# Generate data
data = generate_synthetic_features(n_bars=5000, n_assets=5)

print(f"Data shape: {data.shape}")
print(f"Date range: {data['timestamp'].min()} to {data['timestamp'].max()}")
print(f"\nFirst 5 rows:")
display(data.head())

print(f"\nFeature columns (first 10): {list(data.columns[10:20])}")

---
## 2. Initialize a Random Chromosome

In [ ]:
# Initialize one random chromosome
np.random.seed(123)
random.seed(123)

chromo = Chromosome(n_assets=5)

print("="*60)
print("CHROMOSOME STRUCTURE")
print("="*60)

print("\n[HEAD 1: DIRECTION]")
print(f"Threshold: {chromo.dir_threshold:.3f}")
print("\nGenes:")
for i, gene in enumerate(chromo.dir_genes):
    if gene.active:
        print(f"  {i+1}. Asset={gene.asset}, Feature={gene.feature_id}, "
              f"Lag={gene.lag}, Weight={gene.weight:+.3f}")

print("\n[HEAD 2: SEQUENCE]")
print("\nGenes (showing first 3):")
for i, gene in enumerate(chromo.seq_genes[:3]):
    if gene.active:
        print(f"  {i+1}. Asset={gene.asset}, Feature={gene.feature_id}, Lag={gene.lag}")
        print(f"     Weights: entry_price={gene.weights['entry_price']:+.2f}, "
              f"target_price={gene.weights['target_price']:+.2f}, "
              f"stop_price={gene.weights['stop_price']:+.2f}")

print("\n[HEAD 3: CONFIDENCE]")
print(f"Risk base: {chromo.risk_base*100:.2f}%")
print("\nGenes:")
for i, gene in enumerate(chromo.conf_genes):
    if gene.active:
        print(f"  {i+1}. Asset={gene.asset}, Feature={gene.feature_id}, "
              f"Lag={gene.lag}, Weight={gene.weight:+.3f}")

print("\n[META]")
print(f"Min R:R: {chromo.min_rr:.2f}")

# Validate
errors = validate_chromosome(chromo)
if errors:
    print(f"\n⚠ Validation errors: {errors}")
else:
    print("\n✓ Chromosome is valid")

---
## 3. Generate a Trade Signal

In [ ]:
# Pick a random bar
test_idx = 500
test_bar = data.iloc[test_idx]

print(f"Testing signal generation at bar {test_idx}")
print(f"Timestamp: {test_bar['timestamp']}")
print(f"Current price: {test_bar['close']:.5f}")
print(f"ATR: {test_bar['atr']:.5f}")

# Generate signal
signal = chromo.generate_signal(test_bar)

if signal is None:
    print("\n❌ No trade signal generated (neutral or failed filters)")
else:
    print("\n" + "="*60)
    print("TRADE SIGNAL GENERATED")
    print("="*60)
    
    print(f"\nDirection: {'LONG' if signal.direction == 1 else 'SHORT'}")
    
    print("\nSequence:")
    print(f"  Entry Price:  {signal.sequence['entry']['price']:.5f} "
          f"(in {signal.sequence['entry']['time_expected']} bars)")
    print(f"  Target Price: {signal.sequence['target']['price']:.5f} "
          f"(in {signal.sequence['target']['time_expected']} bars after entry)")
    print(f"  Stop Price:   {signal.sequence['stop']['price']:.5f}")
    print(f"  Expiry:       {signal.sequence['expiry']} bars total")
    
    print("\nConfidence:")
    print(f"  Score:        {signal.confidence['score']:.3f}")
    print(f"  Prob Success: {signal.confidence['prob_success']:.1%}")
    print(f"  Risk %:       {signal.confidence['risk_pct']*100:.2f}%")
    
    print("\nMetrics:")
    print(f"  R:R Ratio:    {signal.rr:.2f}")
    print(f"  Risk (pips):  {abs(signal.sequence['entry']['price'] - signal.sequence['stop']['price'])*10000:.1f}")
    print(f"  Reward (pips): {abs(signal.sequence['target']['price'] - signal.sequence['entry']['price'])*10000:.1f}")
    
    # Validate signal
    sig_errors = validate_trade_signal(signal)
    if sig_errors:
        print(f"\n⚠ Signal validation errors: {sig_errors}")
    else:
        print("\n✓ Signal is valid")

---
## 4. Visualize Signal on Chart

In [ ]:
if signal is not None:
    # Plot 100 bars around signal
    chart_start = max(0, test_idx - 50)
    chart_end = min(len(data), test_idx + 100)
    chart_data = data.iloc[chart_start:chart_end]
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Price
    ax.plot(chart_data['timestamp'], chart_data['close'], 
            label='Close', color='black', linewidth=1.5)
    
    # Signal bar
    ax.axvline(test_bar['timestamp'], color='blue', linestyle='--', 
               alpha=0.7, label='Signal Time')
    
    # Entry
    entry_time = test_bar['timestamp'] + timedelta(minutes=signal.sequence['entry']['time_expected'])
    ax.axhline(signal.sequence['entry']['price'], color='orange', 
               linestyle='--', alpha=0.7, label='Entry')
    ax.axvline(entry_time, color='orange', linestyle=':', alpha=0.5)
    
    # Target
    ax.axhline(signal.sequence['target']['price'], color='green', 
               linestyle='--', linewidth=2, label='Target')
    
    # Stop
    ax.axhline(signal.sequence['stop']['price'], color='red', 
               linestyle='--', linewidth=2, label='Stop')
    
    # Expiry
    expiry_time = test_bar['timestamp'] + timedelta(minutes=signal.sequence['expiry'])
    ax.axvline(expiry_time, color='purple', linestyle=':', 
               alpha=0.5, label='Expiry')
    
    ax.set_xlabel('Time')
    ax.set_ylabel('Price')
    ax.set_title(f"Trade Signal Visualization - {'LONG' if signal.direction == 1 else 'SHORT'}")
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No signal to visualize")

---
## 5. Test Genetic Operators

### 5.1 Crossover

In [ ]:
# Create two parent chromosomes
parent1 = Chromosome(n_assets=5)
parent2 = Chromosome(n_assets=5)

print("Parent 1 - Direction HEAD weights:")
print([g.weight for g in parent1.dir_genes if g.active])

print("\nParent 2 - Direction HEAD weights:")
print([g.weight for g in parent2.dir_genes if g.active])

# Crossover
child = crossover_uniform(parent1, parent2)

print("\nChild - Direction HEAD weights (mix of parents):")
print([g.weight for g in child.dir_genes if g.active])

print("\nChild threshold (average):")
print(f"Parent1: {parent1.dir_threshold:.3f}")
print(f"Parent2: {parent2.dir_threshold:.3f}")
print(f"Child:   {child.dir_threshold:.3f}")

# Validate
errors = validate_chromosome(child)
print(f"\nChild valid: {len(errors) == 0}")

### 5.2 Mutation

In [ ]:
# Clone chromosome
original = copy.deepcopy(child)
mutated = mutate(copy.deepcopy(child), mutation_rate=0.3)

print("Mutation effects:")
print("\nDirection HEAD - Weight changes:")
for i in range(len(original.dir_genes)):
    orig_w = original.dir_genes[i].weight
    mut_w = mutated.dir_genes[i].weight
    if abs(orig_w - mut_w) > 0.001:
        print(f"  Gene {i}: {orig_w:+.3f} → {mut_w:+.3f} (Δ={mut_w-orig_w:+.3f})")

print("\nSequence HEAD - First gene weight changes:")
orig_seq = original.seq_genes[0].weights
mut_seq = mutated.seq_genes[0].weights
for dim in orig_seq:
    if abs(orig_seq[dim] - mut_seq[dim]) > 0.001:
        print(f"  {dim}: {orig_seq[dim]:+.3f} → {mut_seq[dim]:+.3f}")

print(f"\nThreshold: {original.dir_threshold:.3f} → {mutated.dir_threshold:.3f}")
print(f"Risk base: {original.risk_base*100:.2f}% → {mutated.risk_base*100:.2f}%")

---
## 6. Fitness Evaluation

In [ ]:
# Evaluate fitness on first 2000 bars
train_data = data.iloc[:2000]

print("Evaluating chromosome fitness...")
print(f"Training data: {len(train_data)} bars")

fit_score = fitness(chromo, train_data)

print(f"\nFitness Score: {fit_score:.2f}")

# Get detailed stats
stats = chromo.backtest_stats

if stats:
    print("\n" + "="*60)
    print("BACKTEST STATISTICS")
    print("="*60)
    print(f"\nTotal Trades:     {stats['n_trades']}")
    print(f"Wins:             {stats['n_wins']} ({stats['winrate']:.1%})")
    print(f"Losses:           {stats['n_losses']}")
    print(f"No Entry:         {stats['n_no_entry']}")
    print(f"\nTotal PnL:        {stats['total_pnl']:.4f}")
    print(f"Average Win:      {stats['avg_win']:.4f}")
    print(f"Average Loss:     {stats['avg_loss']:.4f}")
    print(f"Avg R:R:          {stats['avg_rr']:.2f}")
    print(f"\nExpectancy:       {stats['expectancy']:.4f}")
    print(f"Sharpe Ratio:     {stats['sharpe']:.2f}")
    print(f"Max Drawdown:     {stats['max_dd']:.4f}")

---
## 7. Population Initialization

In [ ]:
# Initialize population
pop_size = 20
population = initialize_population(pop_size=pop_size, n_assets=5)

print(f"Initialized population of {len(population)} chromosomes")

# Evaluate all
print("\nEvaluating population fitness...")
fitnesses = []

for i, chromo in enumerate(population):
    fit = fitness(chromo, train_data)
    fitnesses.append(fit)
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{pop_size} evaluated")

fitnesses = np.array(fitnesses)

print("\n" + "="*60)
print("POPULATION STATISTICS")
print("="*60)
print(f"\nBest Fitness:    {fitnesses.max():.2f}")
print(f"Worst Fitness:   {fitnesses.min():.2f}")
print(f"Average:         {fitnesses.mean():.2f}")
print(f"Std Dev:         {fitnesses.std():.2f}")
print(f"Median:          {np.median(fitnesses):.2f}")

# Plot distribution
plt.figure(figsize=(10, 5))
plt.hist(fitnesses, bins=15, edgecolor='black', alpha=0.7)
plt.axvline(fitnesses.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {fitnesses.mean():.2f}')
plt.axvline(fitnesses.max(), color='green', linestyle='--', 
            linewidth=2, label=f'Best: {fitnesses.max():.2f}')
plt.xlabel('Fitness')
plt.ylabel('Count')
plt.title('Initial Population Fitness Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 8. Mini Evolution Loop (10 Generations)

In [ ]:
from phase3_Chromosome_Architecture import evolve

# Run small evolution
print("Starting mini-evolution (10 generations)...\n")

best_chromo, history = evolve(
    train_data=train_data,
    val_data=None,
    pop_size=20,
    generations=10,
    elite_pct=0.2,
    mutation_rate=0.15,
    crossover_rate=0.8,
    tournament_size=3
)

print("\n" + "="*60)
print("EVOLUTION COMPLETE")
print("="*60)

---
## 9. Visualize Evolution Progress

In [ ]:
generations = range(len(history['best']))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fitness over time
ax = axes[0, 0]
ax.plot(generations, history['best'], 'o-', label='Best', linewidth=2, markersize=6)
ax.plot(generations, history['avg'], 's-', label='Average', alpha=0.7, markersize=4)
ax.fill_between(
    generations,
    np.array(history['avg']) - np.array(history['std']),
    np.array(history['avg']) + np.array(history['std']),
    alpha=0.2,
    label='±1 std'
)
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness')
ax.set_title('Fitness Evolution')
ax.legend()
ax.grid(True, alpha=0.3)

# Improvement per generation
ax = axes[0, 1]
improvement = np.diff(history['best'])
colors = ['green' if x > 0 else 'red' for x in improvement]
ax.bar(generations[1:], improvement, color=colors, alpha=0.6)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Δ Fitness')
ax.set_title('Improvement per Generation')
ax.grid(True, alpha=0.3)

# Convergence (best - avg)
ax = axes[1, 0]
diversity = np.array(history['best']) - np.array(history['avg'])
ax.plot(generations, diversity, 'o-', color='purple', linewidth=2)
ax.set_xlabel('Generation')
ax.set_ylabel('Best - Average')
ax.set_title('Population Diversity (Convergence)')
ax.grid(True, alpha=0.3)

# Standard deviation
ax = axes[1, 1]
ax.plot(generations, history['std'], 'o-', color='orange', linewidth=2)
ax.set_xlabel('Generation')
ax.set_ylabel('Std Dev of Fitness')
ax.set_title('Population Variance')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal best fitness: {history['best'][-1]:.2f}")
print(f"Improvement: {history['best'][-1] - history['best'][0]:.2f} "
      f"({((history['best'][-1] / history['best'][0]) - 1) * 100:+.1f}%)")

---
## 10. Analyze Best Chromosome

In [ ]:
print("="*60)
print("BEST CHROMOSOME ANALYSIS")
print("="*60)

# Direction HEAD
print("\n[DIRECTION HEAD]")
print(f"Threshold: {best_chromo.dir_threshold:.3f}")
print("\nActive Genes:")
for i, gene in enumerate(best_chromo.dir_genes):
    if gene.active:
        # Get feature name (simplified)
        feat_name = f"feat_{gene.feature_id}"
        print(f"  Gene {i+1}: Asset {gene.asset}, {feat_name}, "
              f"Lag {gene.lag}, Weight {gene.weight:+.3f}")

# Sequence HEAD (top 5 genes)
print("\n[SEQUENCE HEAD] (Top 5 active genes)")
active_seq = [g for g in best_chromo.seq_genes if g.active]
for i, gene in enumerate(active_seq[:5]):
    print(f"\n  Gene {i+1}: Asset {gene.asset}, feat_{gene.feature_id}, Lag {gene.lag}")
    print(f"    entry_price:  {gene.weights['entry_price']:+.2f}")
    print(f"    target_price: {gene.weights['target_price']:+.2f}")
    print(f"    stop_price:   {gene.weights['stop_price']:+.2f}")
    print(f"    entry_time:   {gene.weights['entry_time']:+.2f}")
    print(f"    target_time:  {gene.weights['target_time']:+.2f}")

# Confidence HEAD
print("\n[CONFIDENCE HEAD]")
print(f"Risk Base: {best_chromo.risk_base*100:.2f}%")
print("\nActive Genes:")
for i, gene in enumerate(best_chromo.conf_genes):
    if gene.active:
        print(f"  Gene {i+1}: Asset {gene.asset}, feat_{gene.feature_id}, "
              f"Lag {gene.lag}, Weight {gene.weight:+.3f}")

# Backtest stats
if hasattr(best_chromo, 'backtest_stats') and best_chromo.backtest_stats:
    stats = best_chromo.backtest_stats
    print("\n" + "="*60)
    print("PERFORMANCE METRICS")
    print("="*60)
    print(f"\nTrades:      {stats['n_trades']}")
    print(f"Winrate:     {stats['winrate']:.1%}")
    print(f"Avg R:R:     {stats['avg_rr']:.2f}")
    print(f"Expectancy:  {stats['expectancy']:.4f}")
    print(f"Sharpe:      {stats['sharpe']:.2f}")
    print(f"Max DD:      {stats['max_dd']:.4f}")

---
## 11. Test Best Chromosome on Validation Data

In [ ]:
# Use bars 2000-4000 as validation
val_data = data.iloc[2000:4000]

print(f"Validating on {len(val_data)} unseen bars...")

val_fitness = fitness(best_chromo, val_data)
val_stats = best_chromo.backtest_stats

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(f"\nValidation Fitness: {val_fitness:.2f}")
print(f"Training Fitness:   {history['best'][-1]:.2f}")
print(f"Ratio (val/train):  {val_fitness / history['best'][-1]:.2f}")

if val_stats:
    print(f"\nValidation Trades:  {val_stats['n_trades']}")
    print(f"Winrate:            {val_stats['winrate']:.1%}")
    print(f"Sharpe:             {val_stats['sharpe']:.2f}")
    print(f"Expectancy:         {val_stats['expectancy']:.4f}")
    
    if val_fitness < 0.5 * history['best'][-1]:
        print("\n⚠ WARNING: Significant performance drop on validation (possible overfitting)")
    else:
        print("\n✓ Validation performance acceptable")

---
## 12. Generate and Visualize Sample Trades from Best Chromosome

In [ ]:
# Generate trades on validation data
sample_trades = []

for i in range(len(val_data) - 120):
    bar = val_data.iloc[i]
    signal = best_chromo.generate_signal(bar)
    
    if signal:
        sample_trades.append({
            'timestamp': bar['timestamp'],
            'signal': signal,
            'bar_index': i
        })
    
    if len(sample_trades) >= 5:  # Get first 5 trades
        break

print(f"Generated {len(sample_trades)} sample trades\n")

# Visualize each trade
for idx, trade in enumerate(sample_trades):
    signal = trade['signal']
    bar_idx = trade['bar_index']
    
    print(f"\nTrade {idx+1}: {trade['timestamp']}")
    print(f"  Direction: {'LONG' if signal.direction == 1 else 'SHORT'}")
    print(f"  Entry: {signal.sequence['entry']['price']:.5f} (in {signal.sequence['entry']['time_expected']} bars)")
    print(f"  Target: {signal.sequence['target']['price']:.5f}")
    print(f"  Stop: {signal.sequence['stop']['price']:.5f}")
    print(f"  R:R: {signal.rr:.2f}")
    print(f"  Confidence: {signal.confidence['score']:.2f}")
    print(f"  Risk: {signal.confidence['risk_pct']*100:.2f}%")

print(f"\nVisualize trade #1? (yes/no): ", end="")

---
## 13. Export Best Chromosome

In [ ]:
import pickle
import json

# Save as pickle
with open('best_chromosome.pkl', 'wb') as f:
    pickle.dump(best_chromo, f)
print("✓ Saved to best_chromosome.pkl")

# Save readable JSON summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'fitness': float(history['best'][-1]),
    'validation_fitness': float(val_fitness),
    'generations': len(history['best']),
    'stats': best_chromo.backtest_stats,
    'parameters': {
        'dir_threshold': float(best_chromo.dir_threshold),
        'risk_base': float(best_chromo.risk_base),
        'min_rr': float(best_chromo.min_rr),
        'n_dir_genes': sum(g.active for g in best_chromo.dir_genes),
        'n_seq_genes': sum(g.active for g in best_chromo.seq_genes),
        'n_conf_genes': sum(g.active for g in best_chromo.conf_genes)
    }
}

with open('best_chromosome_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("✓ Saved summary to best_chromosome_summary.json")

print("\n" + "="*60)
print("EXPLORATION COMPLETE")
print("="*60)